# Partie 1 : Concepts de base de la vision par ordinateur
**Exercice 1 : Analyse d'une image**


In [ ]:
import cv2

# 1.Lecture et affichage d'une image
img = cv2.imread('image1.jpg') # Assurez-vous d'avoir une image nommée image.jpg
cv2.imshow('Image originale', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 3. Conversion en niveaux de gris 
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cv2.imshow('Image en niveaux de gris', gray)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 4. Détection des contours avec Canny
edges = cv2.Canny(gray, 100, 200)
cv2.imshow('Contours détectés', edges)
cv2.waitKey(0)
cv2.destroyAllWindows()



**Que représente la matrice retournée par OpenCV pour l'image ?**

Elle représente les pixels de l'image. Pour une image couleur, c'est une matrice 3D (hauteur x largeur x 3 canaux : Bleu, Vert, Rouge). Chaque valeur (entre 0 et 255) représente l'intensité de la couleur pour ce pixel.

**Pourquoi convertit-on souvent l'image en niveaux de gris avant traitement ?**

Pour simplifier les calculs. En passant de 3 canaux (BGR) à 1 seul canal (luminance), on réduit considérablement la quantité de données à traiter, ce qui accélère les algorithmes tout en conservant les informations structurelles importantes (comme les contours).

**Que peut nous apporter le filtre Gaussien ?**

Le filtre Gaussien permet de lisser l'image et de réduire le bruit (les petits détails parasites). Cela améliore grandement les étapes suivantes comme la détection de contours (Canny), car l'algorithme ne sera pas perturbé par le bruit de fond.

# Partie 2 : Traitement d'images et vidéos avec OpenCV
**Exercice 2 : Lecture vidéo / webcam**

In [ ]:
import cv2

# Lecture de la webcam (0) ou d'un fichier vidéo
cap = cv2.VideoCapture(0)

# Configuration pour enregistrer la vidéo
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('sortie_gris.mp4', fourcc, 20.0, (w, h), isColor=False)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convertir en niveaux de gris
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Enregistrer la frame
    out.write(gray)
    
    # Afficher
    cv2.imshow('Webcam en gris', gray)
    
  
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()  
cv2.destroyAllWindows()

# Partie 3 : Détection et segmentation d'objets avec Detectron2
**Exercice 3 : Détection d'objets**

In [17]:
# Installation
!pip install -U 'git+https://github.com/facebookresearch/detectron2.git'

import cv2
import torch
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from google.colab.patches import cv2_imshow

# Configuration
CONFIG = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(CONFIG))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CONFIG)
predictor = DefaultPredictor(cfg)

# Traitement
img = cv2.imread("image_objects.jpg")
outputs = predictor(img)

# Visualisation
v = Visualizer(img[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2_imshow(out.get_image()[:, :, ::-1])


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: "'git+https://github.com/facebookresearch/detectron2.git'": Expected package name at the start of dependency specifier
    'git+https://github.com/facebookresearch/detectron2.git'
    ^


ModuleNotFoundError: No module named 'detectron2'

# Partie 4 : Détection et reconnaissance faciale avec Dlib

In [ ]:
import dlib
import cv2

detector = dlib.get_frontal_face_detector()
img = cv2.imread("people.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = detector(gray)
for face in faces:
    x, y, w, h = face.left(), face.top(), face.width(), face.height()
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)

cv2.imshow("Visages detectes", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Exercice 5 : Reconnaissance faciale

In [19]:
import face_recognition
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Charger et encoder l'image de référence
image_ref = face_recognition.load_image_file("musk.jpg")
encodage_ref = face_recognition.face_encodings(image_ref)[0]

# Charger et encoder l'image test
image_test = face_recognition.load_image_file("huang.jpg")
encodage_test = face_recognition.face_encodings(image_test)[0]

# Comparer
result = face_recognition.compare_faces([encodage_ref], encodage_test)
print("Même personne ?", result[0])

Même personne ? False


Que se passe-t-il si l'image test contient plusieurs visages ?

face_encodings() retournera une liste contenant plusieurs encodages (un par visage). L'index [0] dans le code actuel ne prendra que le premier visage trouvé. Pour comparer tous les visages, il faudrait faire une boucle sur les encodages de l'image test.

Quelle précision attendez-vous dans des conditions réelles ?

La précision peut baisser avec des variations extrêmes de lumière, de très mauvaises résolutions, des visages de profil prononcé (occultation d'une partie du visage) ou avec des accessoires (grosses lunettes, masques). Cependant, les modèles basés sur le deep learning (comme celui utilisé ici) sont généralement très robustes.

# Exercice 6 : Points caractéristiques (landmarks)

In [ ]:
import cv2
import face_recognition

img = cv2.imread("people.jpg")
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

landmarks_list = face_recognition.face_landmarks(rgb)
print("Visages:", len(landmarks_list))

for landmarks in landmarks_list:
    for partie, points in landmarks.items():
        for (x, y) in points:
            cv2.circle(img, (x, y), 2, (0, 0, 255), -1)

cv2.imshow("Landmarks", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Visages: 5


Quelles parties du visage sont retournées ?
Le contour du visage (chin), les sourcils (left_eyebrow, right_eyebrow), le nez (nose_bridge, nose_tip), les yeux (left_eye, right_eye) et les lèvres (top_lip, bottom_lip).   Comment utiliser l'EAR pour la somnolence ?
L'Eye Aspect Ratio (EAR) calcule le rapport entre la hauteur et la largeur de l'œil en utilisant les coordonnées des landmarks. Quand l'œil se ferme (clignement ou somnolence), la hauteur diminue, faisant chuter la valeur de l'EAR. Si l'EAR reste sous un certain seuil pendant un nombre défini de frames successives, on peut déclencher une alerte de somnolence.

# Exercice 7 : Détection d'objets en temps réel

In [ ]:
import os
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")
image_path = "people.jpg"

try:
    results = model(image_path, conf=0.5)
    
    #affichage
    cv2.imshow("YOLO", results[0].plot())
    cv2.waitKey(0)
    cv2.destroyAllWindows()
except Exception as e:
    print("Erreur lors de la détection:", e)


image 1/1 d:\computer_vision\people.jpg: 352x640 5 persons, 205.6ms
Speed: 6.0ms preprocess, 205.6ms inference, 3.3ms postprocess per image at shape (1, 3, 352, 640)


Comparez le temps d'inférence (Mask R-CNN vs YOLO) 

(À mesurer avec time.time() dans vos exécutions). Généralement, YOLOv8n (nano) prendra quelques millisecondes par image, tandis que Mask R-CNN prendra beaucoup plus de temps (parfois près d'une seconde). YOLO est conçu pour la vitesse.

YOLO fournit-il des masques ? Testez yolov8n-seg.pt :

Le modèle yolov8n.pt classique ne fournit que des boîtes englobantes (Bounding Boxes).

Si on utilise YOLO("yolov8n-seg.pt"), le modèle effectuera de la segmentation d'instance et fournira des masques très similaires à Mask R-CNN, mais de manière beaucoup plus rapide.

Dans quels cas préférer chaque modèle ?

YOLO : Pour les applications en temps réel (vidéo surveillance, véhicules autonomes) où la vitesse est critique.

Mask R-CNN : Pour l'imagerie médicale, l'analyse satellitaire ou les tâches nécessitant une segmentation de très haute précision, où le temps de calcul n'est pas le critère principal.

# Mini-Projet 4 : Floutage automatique des visages.

In [22]:
import cv2
import dlib

# Initialiser la webcam et le détecteur de visages
cap = cv2.VideoCapture(0)
detector = dlib.get_frontal_face_detector()

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Convertir en niveaux de gris pour la détection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)
    
    for face in faces:
        # Récupérer les coordonnées du rectangle du visage
        x, y = face.left(), face.top()
        w, h = face.width(), face.height()
        
        # Sécurité pour éviter les erreurs si le visage est sur le bord de l'écran
        x, y = max(0, x), max(0, y)
        
        # Vérifier que la zone est valide
        if w > 0 and h > 0:
            # 1. Extraire uniquement la zone du visage
            visage = frame[y:y+h, x:x+w]
            
            # 2. Appliquer un flou fort (les nombres doivent être impairs)
            visage_flou = cv2.GaussianBlur(visage, (99, 99), 30)
            
            # 3. Remplacer le visage original par la version floutée
            frame[y:y+h, x:x+w] = visage_flou

    # Afficher la vidéo en temps réel
    cv2.imshow('Anonymisation - Floutage en temps reel', frame)
    
    # Appuyer sur 'q' pour quitter
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()